In [ ]:
!pip install -q transformers[torch] datasets evaluate seqeval sentencepiece accelerate protobuf 
!pip install -q -U bitsandbytes

In [ ]:
import yaml
import numpy as np
import evaluate
from datasets import load_from_disk, load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    DataCollatorForTokenClassification, 
    TrainingArguments, 
    Trainer
)

import os

In [ ]:
DATA_PATH = os.path.join(os.path.dirname(__file__), 'processed', 'multinerd_en')

In [ ]:
with open(DATA_PATH + "config.yml", "r") as f:
    config = yaml.safe_load(f)

label2id = config['labels']['label2id']
id2label = {v: k for k, v in label2id.items()}
label_list = list(label2id.keys())

In [ ]:
dataset = load_from_disk(DATA_PATH)

In [ ]:
dataset['train'][0]

{'tokens': ['Herbs',
  'can',
  'be',
  'perennials',
  'such',
  'as',
  'thyme',
  ',',
  'sage',
  'or',
  'lavender',
  ',',
  'biennials',
  'such',
  'as',
  'parsley',
  ',',
  'or',
  'annuals',
  'like',
  'basil',
  '.'],
 'ner_tags': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  25,
  0,
  0,
  0,
  0,
  25,
  0],
 'lang': 'en'}

### PREPARE TOKENIZER, METRIC

In [11]:
model_id = "FacebookAI/xlm-roberta-base"

In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
enc = tokenizer(["EU", "rejects"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
word_ids = enc.word_ids()

for tok, wid in zip(tokens, word_ids):
    print(f"{tok:15s}  word_id={wid}")

<s>              word_id=None
▁EU              word_id=0
▁re              word_id=1
ject             word_id=1
s                word_id=1
</s>             word_id=None


In [ ]:
def tokenize_and_align_labels(examples):
    # 1. Để tokenizer tự thêm <s>, </s> (tương đương CLS, SEP) và padding
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=config['model']['max_length'],
        add_special_tokens=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            # Token đặc biệt (<s>, </s>, <pad>) có word_id là None
            if word_idx is None:
                label_ids.append(-100)
            # Nếu là sub-token đầu tiên của một từ
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Nếu là các sub-token sau của cùng một từ
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
# Tokenize all dataset
tokenized_ds = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/131280 [00:00<?, ? examples/s]

Map:   0%|          | 0/16410 [00:00<?, ? examples/s]

Map:   0%|          | 0/16454 [00:00<?, ? examples/s]

In [17]:
tokenized_ds.set_format(
    type='torch', 
    columns=['input_ids', 'attention_mask', 'labels']
)

In [18]:
print(tokenized_ds["train"][0])

{'input_ids': tensor([    0,  1840, 16145,   831,   186,   117,    33,    93, 16765,  6044,
          237, 96055,   282,     6,     4,    57,   429,   707, 28778,  7944,
            6,     4,  1806,    93, 16765,  6044,   237,   366,     7,  4293,
            6,     4,   707, 77253,     7,  1884,  4950,   379,     6,     5,
            2]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), 'labels': tensor([-100,    0, -100,    0,    0,    0, -100, -100, -100,    0,    0,    0,
        -100,    0, -100,    0, -100,    0,    0, -100,    0, -100,    0, -100,
        -100,    0,    0,   25, -100, -100,    0, -100,    0,    0, -100,    0,
          25, -100,    0, -100, -100])}


In [19]:
# 3. Định nghĩa Metrics (F1-score)
metric = evaluate.load("seqeval")

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "accuracy": results["overall_accuracy"],
    }

### TRAINING MODEL

In [21]:
model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
args = TrainingArguments(
    output_dir="./xlm-roberta-results",
    eval_strategy="steps",
    eval_steps=200, 
    logging_steps=200,
    save_strategy="steps",       # THÊM MỚI: Phải giống với eval_strategy
    save_steps=200,
    
    learning_rate=1e-5,
    max_grad_norm=1.0,
    per_device_train_batch_size=config['training']['batch_size'],
    num_train_epochs=config['training']['epochs'],
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",   # Chọn tiêu chí để đánh giá "tốt nhất" (có thể dùng "eval_loss")
    greater_is_better=False,
    save_total_limit=2,
    
    fp16=False,                # Tạm thời tắt fp16 nếu vẫn bị lỗi trên 2 GPU
    gradient_accumulation_steps=2, 
    dataloader_num_workers=2,   # Tối ưu cho Kaggle
    report_to="none",
    optim="paged_adamw_8bit",

    ddp_find_unused_parameters=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

In [23]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Accuracy
200,2.162488,0.254689,0.968512
400,0.409370,0.165907,0.975080
600,0.300040,0.140345,0.977641
800,0.263734,0.128815,0.978832
1000,0.236169,0.123624,0.978930
1200,0.217148,0.120059,0.978998
1400,0.203299,0.116382,0.979675
1600,0.204492,0.111979,0.980562
1800,0.189671,0.111236,0.980311
2000,0.188032,0.107485,0.980852


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3078, training_loss=0.3448422384231101, metrics={'train_runtime': 6986.0639, 'train_samples_per_second': 56.375, 'train_steps_per_second': 0.441, 'total_flos': 1.7233208618986464e+16, 'train_loss': 0.3448422384231101, 'epoch': 3.0})

In [25]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Lấy token từ hệ thống Secret của Kaggle
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf-token") # Thay "HF_TOKEN" bằng tên bạn đã đặt

# 2. Đăng nhập vào Hugging Face bằng token đó
login(token=hf_token)

In [ ]:
repo_id = "imbee510/finetuned_multiner_xlm_roberta" 

# Đẩy mô hình và tokenizer lên
# Nếu repo chưa tồn tại, nó sẽ tự tạo mới cho bạn
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Xong rồi! Check hàng tại: https://huggingface.co/imbee510/finetuned_multiner_xlm_roberta
